<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/fluidflow/neqsim_fenicsx_fem_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeqSim + FEniCSx: local FEM from process-model boundary conditions

This advanced example connects **NeqSim Java master → fluid properties / hydrate equilibrium → 1D pipeline screen → FEniCSx axisymmetric wall/insulation FEM → cooldown / hydrate margin → thermo-elastic stress**. The case is a wet-gas subsea line with a local degraded-insulation patch. It is a teaching and screening model, not a piping-code assessment.

The notebook deliberately keeps NeqSim responsible for thermodynamics and process/flow boundary conditions while FEniCSx resolves local solid fields where geometry and material gradients matter.

In [ ]:
import hashlib, importlib.metadata, os, shutil, subprocess, sys
from pathlib import Path

NEQSIM_SOURCE_REF = 'master'
os.environ['NEQSIM_JVM_AUTOSTART'] = '0'

def rq(cmd, cwd=None):
    return subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=True).stdout

rq([sys.executable, '-m', 'pip', 'install', '-q', 'neqsim', 'scipy'])
try:
    import dolfinx
except ImportError:
    installer = Path('/tmp/fenicsx.sh')
    rq(['wget', '-q', 'https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh', '-O', str(installer)])
    rq(['bash', str(installer)])
    import dolfinx

src = Path('/content/neqsim-java')
if src.exists():
    shutil.rmtree(src)
rq(['git', 'clone', '--depth', '1', '--branch', NEQSIM_SOURCE_REF, 'https://github.com/equinor/neqsim.git', str(src)])
commit = rq(['git', '-C', str(src), 'rev-parse', 'HEAD']).strip()
rq(['./mvnw', '-q', '-DskipTests', '-P', 'shade', 'package'], cwd=src)
candidates = [p for p in (src / 'target').glob('neqsim-*.jar') if '-sources' not in p.name and '-javadoc' not in p.name and not p.name.startswith('original-')]
if not candidates:
    raise FileNotFoundError('No NeqSim runtime JAR found')
jar = max(candidates, key=lambda p: p.stat().st_size)
assert jar.stat().st_size > 5_000_000
jar_sha = hashlib.sha256(jar.read_bytes()).hexdigest()

import jpype
jpype.addClassPath(str(jar))
if not jpype.isJVMStarted():
    jpype.startJVM('-Xrs', convertStrings=False, interrupt=False)

SystemSrkCPAstatoil = jpype.JClass('neqsim.thermo.system.SystemSrkCPAstatoil')
ThermodynamicOperations = jpype.JClass('neqsim.thermodynamicoperations.ThermodynamicOperations')
SurfCooldownAnalyzer = jpype.JClass('neqsim.pvtsimulation.flowassurance.SurfCooldownAnalyzer')
loc = str(SurfCooldownAnalyzer.class_.getProtectionDomain().getCodeSource().getLocation())
assert jar.name in loc

print('NeqSim master commit:', commit)
print('JAR SHA-256:', jar_sha)
print('Main-only class source:', loc)
print('DOLFINx:', dolfinx.__version__)

## 1. NeqSim fluid → heat-transfer boundary

NeqSim supplies phase equilibrium, density, viscosity, thermal conductivity and mass-specific heat capacity. A Gnielinski correlation converts these into an internal convection coefficient. A 1D energy balance then establishes the local bulk-gas temperature and pressure used as the FEM boundary condition.

This is the main modeling handoff: **the FEM model does not invent fluid properties**.

In [ ]:
from mpi4py import MPI
from petsc4py import PETSc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

assert MPI.COMM_WORLD.size == 1

Tin, Pin, Pout, Tsea = 60.0, 80.0, 70.0, 4.0
Di, ts, ti = 0.254, 0.0127, 0.050
ri, rs, ro = Di / 2, Di / 2 + ts, Di / 2 + ts + ti
Lpipe, velocity = 20_000.0, 5.0
ks, kins, kbad = 50.0, 0.17, 0.70
ho = 300.0
localL, z0, z1 = 4.0, 1.5, 2.5
rhos, cps, rhoi, cpi = 7850.0, 500.0, 600.0, 1700.0
depth, rhosea, g = 300.0, 1025.0, 9.80665

dry = {'nitrogen':0.01, 'CO2':0.02, 'methane':0.85, 'ethane':0.07, 'propane':0.03, 'i-butane':0.006, 'n-butane':0.008, 'i-pentane':0.002, 'n-pentane':0.002, 'n-hexane':0.002}
water = 2e-4
z = {name: value * (1.0 - water) for name, value in dry.items()}
z['water'] = water

def fluid(Tc, Pbara, hydrate=False):
    f = SystemSrkCPAstatoil(Tc + 273.15, Pbara)
    for name, value in z.items():
        f.addComponent(name, float(value))
    f.setMixingRule(10)
    f.setMultiPhaseCheck(True)
    f.setHydrateCheck(bool(hydrate))
    ThermodynamicOperations(f).TPflash()
    f.initPhysicalProperties()
    return f

def props(Tc, Pbara):
    phase = fluid(Tc, Pbara).getPhase('gas')
    return {
        'rho': float(phase.getDensity('kg/m3')),
        'mu': float(phase.getViscosity('kg/msec')),
        'k': float(phase.getThermalConductivity('W/mK')),
        'cp': float(phase.getCp('J/kgK')),
    }

p0 = props(Tin, Pin)
Re = p0['rho'] * velocity * Di / p0['mu']
Pr = p0['cp'] * p0['mu'] / p0['k']
friction = (0.79 * np.log(Re) - 1.64) ** -2
Nu = (friction / 8) * (Re - 1000) * Pr / (1 + 12.7 * np.sqrt(friction / 8) * (Pr ** (2 / 3) - 1))
hi = Nu * p0['k'] / Di

def Rprime(k_ins=kins):
    return (
        1 / (hi * 2 * np.pi * ri)
        + np.log(rs / ri) / (2 * np.pi * ks)
        + np.log(ro / rs) / (2 * np.pi * k_ins)
        + 1 / (ho * 2 * np.pi * ro)
    )

x = np.linspace(0.0, Lpipe, 121)
P = np.linspace(Pin, Pout, len(x))
T = np.empty_like(x)
T[0] = Tin
Aflow = np.pi * ri**2
for j in range(len(x) - 1):
    state = props(T[j], P[j])
    mdot = state['rho'] * velocity * Aflow
    T[j + 1] = T[j] - (1 / Rprime()) * (T[j] - Tsea) / (mdot * state['cp']) * (x[j + 1] - x[j])

Tb = float(np.interp(Lpipe / 2, x, T))
Pb = float(np.interp(Lpipe / 2, x, P))
summary = pd.DataFrame({'rho [kg/m3]':[p0['rho']], 'mu [Pa s]':[p0['mu']], 'k [W/mK]':[p0['k']], 'Cp [J/kgK]':[p0['cp']], 'Re':[Re], 'Pr':[Pr], 'hi [W/m2K]':[hi]})
display(summary)
print(f'10 km FEM boundary: {Tb:.3f} degC, {Pb:.2f} bara')
plt.plot(x / 1000, T)
plt.xlabel('Distance [km]'); plt.ylabel('Bulk-gas temperature [degC]'); plt.grid(); plt.show()

## 2. FEniCSx local thermal field

The local $z-r$ model is axisymmetric, so each weak-form integral is weighted by $2\pi r$. The inner steel wall convects to the NeqSim bulk gas and the outer insulation convects to seawater. A one-metre section uses increased effective insulation conductivity to represent a local degradation/water-ingress sensitivity.

Only one retained `LinearProblem` is used for this steady solve. The far field is checked against the independent analytical cylindrical-resistance solution.

In [ ]:
def tags_rect(domain, rmin, rmax):
    fdim = domain.topology.dim - 1
    inner = mesh.locate_entities_boundary(domain, fdim, lambda xx: np.isclose(xx[1], rmin))
    outer = mesh.locate_entities_boundary(domain, fdim, lambda xx: np.isclose(xx[1], rmax))
    entities = np.hstack([inner, outer]).astype(np.int32)
    values = np.hstack([np.ones(len(inner), np.int32), 2 * np.ones(len(outer), np.int32)])
    order = np.argsort(entities)
    return mesh.meshtags(domain, fdim, entities[order], values[order])

def solve_thermal(nx=100, nr=24):
    domain = mesh.create_rectangle(MPI.COMM_WORLD, np.array([[0.0, ri], [localL, ro]]), [nx, nr], cell_type=mesh.CellType.triangle)
    facet_tags = tags_rect(domain, ri, ro)
    ds = ufl.Measure('ds', domain=domain, subdomain_data=facet_tags)
    dx = ufl.Measure('dx', domain=domain)
    V = fem.functionspace(domain, ('Lagrange', 1))
    u, w = ufl.TrialFunction(V), ufl.TestFunction(V)
    X = ufl.SpatialCoordinate(domain)
    r = X[1]
    base_k = ufl.conditional(ufl.le(r, rs), ks, kins)
    degraded = ufl.And(ufl.And(ufl.ge(X[0], z0), ufl.le(X[0], z1)), ufl.gt(r, rs))
    kval = ufl.conditional(degraded, kbad, base_k)
    a = kval * ufl.inner(ufl.grad(u), ufl.grad(w)) * 2 * np.pi * r * dx + hi * u * w * 2 * np.pi * r * ds(1) + ho * u * w * 2 * np.pi * r * ds(2)
    L = hi * (Tb + 273.15) * w * 2 * np.pi * r * ds(1) + ho * (Tsea + 273.15) * w * 2 * np.pi * r * ds(2)
    problem = LinearProblem(a, L, petsc_options_prefix='heat_main_', petsc_options={'ksp_type':'cg', 'pc_type':'jacobi', 'ksp_rtol':1e-10})
    solution = problem.solve()
    assert problem.solver.getConvergedReason() > 0
    return domain, facet_tags, solution, problem

domain_t, thermal_tags, Th, thermal_problem = solve_thermal()
coords = Th.function_space.tabulate_dof_coordinates()
Tc = Th.x.array - 273.15
inner = np.isclose(coords[:, 1], ri)
z_inner = coords[inner, 0]
T_inner = Tc[inner]
order = np.argsort(z_inner)

q_uniform = (Tb - Tsea) / Rprime()
T_analytic = Tb - q_uniform / (hi * 2 * np.pi * ri)
far = inner & ((coords[:, 0] < 0.55) | (coords[:, 0] > 3.45))
T_far = float(Tc[far].mean())
assert Tsea - 1e-6 <= float(Tc.min()) <= float(Tc.max()) <= Tb + 1e-6
assert abs(T_far - T_analytic) < 1.5
assert float(T_inner.min()) < T_far

print(f'Analytical uniform inner-wall T: {T_analytic:.3f} degC')
print(f'FEM far-field inner-wall T: {T_far:.3f} degC')
print(f'FEM degraded-section minimum: {float(T_inner.min()):.3f} degC')
plt.plot(z_inner[order], T_inner[order], label='FEniCSx inner wall')
plt.axvspan(z0, z1, alpha=0.2, label='degraded insulation')
plt.axhline(T_analytic, ls='--', label='uniform analytical')
plt.xlabel('Local z [m]'); plt.ylabel('Inner-wall T [degC]'); plt.legend(); plt.grid(); plt.show()
plt.figure(figsize=(10, 3))
sc = plt.scatter(coords[:, 0], coords[:, 1], c=Tc, s=10)
plt.colorbar(sc, label='Temperature [degC]')
plt.axvspan(z0, z1, alpha=0.12)
plt.xlabel('Local z [m]'); plt.ylabel('Radius [m]'); plt.title('FEniCSx local temperature field'); plt.show()

## 3. Hydrate no-touch time: NeqSim screening versus local FEM cold spot

NeqSim calculates hydrate equilibrium at the local pressure and its `SurfCooldownAnalyzer` provides a fast uniform-pipe screening model. The local transient FEM resolves the solid thermal storage and the degraded-insulation cold spot.

The transient FEM reuses one assembled `LinearProblem` while only the NeqSim-derived bulk boundary and previous-temperature field change.

In [ ]:
hyd = fluid(Tb, Pb, True)
ThermodynamicOperations(hyd).hydrateFormationTemperature()
Thyd = float(hyd.getTemperature('C'))
target = Thyd + 3.0

screen = SurfCooldownAnalyzer(fluid(Tb, Pb))
screen.setInternalDiameter(Di); screen.setWallThickness(ts); screen.setInsulationThickness(ti)
screen.setInsulationConductivity(kins); screen.setExternalHTC(ho); screen.setSeabedTemperature(Tsea)
screen.setOperatingTemperature(Tb); screen.setHydrateMargin(3.0); screen.setTotalTimeHours(48.0); screen.calculate()
print(f'Hydrate equilibrium: {Thyd:.3f} degC')
print(f'NeqSim lumped no-touch time: {float(screen.getNoTouchTimeHours()):.2f} h')

def cooldown(dt=1800.0, hours=48.0):
    domain = mesh.create_rectangle(MPI.COMM_WORLD, np.array([[0.0, ri], [localL, ro]]), [50, 14], cell_type=mesh.CellType.triangle)
    tags = tags_rect(domain, ri, ro)
    ds = ufl.Measure('ds', domain=domain, subdomain_data=tags); dx = ufl.Measure('dx', domain=domain)
    V = fem.functionspace(domain, ('Lagrange', 1))
    X = ufl.SpatialCoordinate(domain); r = X[1]
    base_k = ufl.conditional(ufl.le(r, rs), ks, kins)
    degraded = ufl.And(ufl.And(ufl.ge(X[0], z0), ufl.le(X[0], z1)), ufl.gt(r, rs))
    kval = ufl.conditional(degraded, kbad, base_k)
    C = ufl.conditional(ufl.le(r, rs), rhos * cps, rhoi * cpi)
    previous = fem.Function(V); previous.x.array[:] = Tb + 273.15
    u, w = ufl.TrialFunction(V), ufl.TestFunction(V)
    bulkK = fem.Constant(domain, PETSc.ScalarType(Tb + 273.15)); seaK = fem.Constant(domain, PETSc.ScalarType(Tsea + 273.15))
    a = C * u * w * 2 * np.pi * r * dx + dt * kval * ufl.inner(ufl.grad(u), ufl.grad(w)) * 2 * np.pi * r * dx + dt * hi * u * w * 2 * np.pi * r * ds(1) + dt * ho * u * w * 2 * np.pi * r * ds(2)
    L = C * previous * w * 2 * np.pi * r * dx + dt * hi * bulkK * w * 2 * np.pi * r * ds(1) + dt * ho * seaK * w * 2 * np.pi * r * ds(2)
    problem = LinearProblem(a, L, petsc_options_prefix='cooldown_main_', petsc_options={'ksp_type':'cg', 'pc_type':'jacobi', 'ksp_rtol':1e-9})
    gp = props(Tb, Pb)
    fluid_capacity = gp['rho'] * np.pi * ri**2 * localL * gp['cp']
    times, bulk, wall = [0.0], [Tb], [Tb]
    coords = V.tabulate_dof_coordinates(); inner = np.isclose(coords[:, 1], ri)
    for n in range(int(hours * 3600 / dt)):
        bulkK.value = PETSc.ScalarType(bulk[-1] + 273.15)
        current = problem.solve()
        assert problem.solver.getConvergedReason() > 0
        q_to_wall = fem.assemble_scalar(fem.form(hi * (bulkK - current) * 2 * np.pi * r * ds(1)))
        bulk.append(float(bulkK.value) - q_to_wall * dt / fluid_capacity - 273.15)
        wall.append(float((current.x.array - 273.15)[inner].min()))
        times.append((n + 1) * dt / 3600.0)
        previous.x.array[:] = current.x.array
    return np.array(times), np.array(bulk), np.array(wall), problem

tc, bc, wc, cooldown_problem = cooldown()
def crossing(series):
    idx = np.where(series <= target)[0]
    return tc[idx[0]] if len(idx) else np.inf

print(f'FEM bulk no-touch time: {crossing(bc):.2f} h')
print(f'FEM local-wall no-touch time: {crossing(wc):.2f} h')
assert np.all(np.diff(bc) <= 1e-6)
plt.plot(tc, bc, label='bulk fluid')
plt.plot(tc, wc, label='minimum inner wall')
plt.axhline(target, ls='--', label='hydrate + 3 K')
plt.xlabel('Time [h]'); plt.ylabel('Temperature [degC]'); plt.legend(); plt.grid(); plt.show()

## 4. Thermo-mechanical stress screen

The steady FEniCSx temperature field is transferred to a steel-only axisymmetric elasticity model. NeqSim supplies internal pressure; seawater depth supplies external pressure. The result illustrates a local thermo-mechanical screening calculation, not a design-code check.

In [ ]:
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator

xy = Th.function_space.tabulate_dof_coordinates()[:, :2]
linear_T = LinearNDInterpolator(xy, Th.x.array)
nearest_T = NearestNDInterpolator(xy, Th.x.array)
def transfer_temperature(xx):
    points = np.c_[xx[0], xx[1]]
    values = linear_T(points)
    missing = ~np.isfinite(values)
    values[missing] = nearest_T(points[missing])
    return values

steel = mesh.create_rectangle(MPI.COMM_WORLD, np.array([[0.0, ri], [localL, rs]]), [80, 8], cell_type=mesh.CellType.triangle)
steel_tags = tags_rect(steel, ri, rs)
ds = ufl.Measure('ds', domain=steel, subdomain_data=steel_tags); dx = ufl.Measure('dx', domain=steel)
VT = fem.functionspace(steel, ('Lagrange', 1)); TT = fem.Function(VT); TT.interpolate(transfer_temperature)
V = fem.functionspace(steel, ('Lagrange', 1, (2,)))
u, w = ufl.TrialFunction(V), ufl.TestFunction(V)
X = ufl.SpatialCoordinate(steel); r = X[1]
E, nu, alpha = 207e9, 0.30, 12e-6
mu = E / (2 * (1 + nu)); lam = E * nu / ((1 + nu) * (1 - 2 * nu)); I3 = ufl.Identity(3)
def eps(v):
    return ufl.as_tensor([[v[0].dx(0), 0.5 * (v[0].dx(1) + v[1].dx(0)), 0], [0.5 * (v[0].dx(1) + v[1].dx(0)), v[1].dx(1), 0], [0, 0, v[1] / r]])
def sigma(v):
    elastic = eps(v) - alpha * (TT - (Tb + 273.15)) * I3
    return 2 * mu * elastic + lam * ufl.tr(elastic) * I3

a = ufl.inner(sigma(u), eps(w)) * 2 * np.pi * r * dx
pi_pressure = Pb * 1e5; po_pressure = rhosea * g * depth
L = ufl.dot(ufl.as_vector((0.0, pi_pressure)), w) * 2 * np.pi * r * ds(1) + ufl.dot(ufl.as_vector((0.0, -po_pressure)), w) * 2 * np.pi * r * ds(2)
fdim = steel.topology.dim - 1
left = mesh.locate_entities_boundary(steel, fdim, lambda xx: np.isclose(xx[0], 0.0))
dofs_z = fem.locate_dofs_topological(V.sub(0), fdim, left)
bc_z = fem.dirichletbc(PETSc.ScalarType(0.0), dofs_z, V.sub(0))
elastic_problem = LinearProblem(a, L, bcs=[bc_z], petsc_options_prefix='elastic_', petsc_options={'ksp_type':'preonly', 'pc_type':'lu'})
displacement = elastic_problem.solve()
assert elastic_problem.solver.getConvergedReason() > 0
S = sigma(displacement); dev = S - ufl.tr(S) / 3 * I3; vm = ufl.sqrt(1.5 * ufl.inner(dev, dev))
V0 = fem.functionspace(steel, ('Discontinuous Lagrange', 0)); q = ufl.TrialFunction(V0); v0 = ufl.TestFunction(V0)
vm_problem = LinearProblem(q * v0 * dx, vm * v0 * dx, petsc_options_prefix='vm_', petsc_options={'ksp_type':'preonly', 'pc_type':'lu'})
vmh = vm_problem.solve()
assert vm_problem.solver.getConvergedReason() > 0

A_lame = (pi_pressure * ri**2 - po_pressure * rs**2) / (rs**2 - ri**2)
B_lame = ri**2 * rs**2 * (pi_pressure - po_pressure) / (rs**2 - ri**2)
hoop_inner_pressure_only = (A_lame + B_lame / ri**2) / 1e6
print(f'Lamé pressure-only inner hoop stress reference: {hoop_inner_pressure_only:.1f} MPa')
print(f'Maximum pressure + thermal von Mises screen: {float(vmh.x.array.max()/1e6):.1f} MPa')

## Model hierarchy and next use cases

Use NeqSim for the thermodynamic/process envelope and fast screening; use local FEM where geometry or material gradients matter. The companion `finite_element_methods_oil_gas_neqsim.ipynb` establishes the wider **Gmsh + scikit-fem + FEniCSx + PyVista** stack and includes porous diffusion and wellbore/formation heat transfer.

Natural extensions are buried-pipeline soil domains, separator/nozzle thermal stress, corrosion/electrochemistry PDEs and coupled geomechanics.